# 🛫 **FEATURE SELECTION - AIRLINE PASSENGER SATISFACTION**

---

**Autor:** Luís Fernando  
**Objetivo:** Aplicar métodos de seleção de atributos (Mutual Information e ANOVA) utilizando SelectKBest e comparar resultados com Diagrama de Venn

**Dataset:** [Airline Passenger Satisfaction](https://www.kaggle.com/datasets/teejmahal20/airline-passenger-satisfaction)

---

## Metodologia:
1. Carregamento e exploração dos dados
2. Pré-processamento (tratamento de valores ausentes, encoding)
3. Feature Selection com Mutual Information
4. Feature Selection com ANOVA (F-classif)
5. Comparação com Diagrama de Venn (K=10 e K=15)
6. Avaliação de modelos preditivos
7. Análise comparativa e conclusões

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalar bibliotecas necessárias
!pip install matplotlib-venn kagglehub -q

# Importações
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2

# Sklearn - Pré-processamento
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Sklearn - Feature Selection
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif

# Sklearn - Modelos
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Sklearn - Métricas
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados

In [ ]:
print("="*70)
print("CARREGAMENTO DO DATASET - AIRLINE PASSENGER SATISFACTION")
print("="*70)

# Download via kagglehub
try:
    import kagglehub
    path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
    print(f"📁 Dataset baixado em: {path}")
    
    # Carregar os arquivos
    import os
    files = os.listdir(path)
    print(f"📄 Arquivos disponíveis: {files}")
    
    # Carregar train e test
    df_train = pd.read_csv(f"{path}/train.csv")
    df_test = pd.read_csv(f"{path}/test.csv")
    
    # Combinar para análise completa
    df = pd.concat([df_train, df_test], ignore_index=True)
    print(f"✅ Dataset carregado: {df.shape[0]} registros, {df.shape[1]} colunas")
    
except Exception as e:
    print(f"⚠️ Erro ao baixar via kagglehub: {e}")
    print("\n📌 ALTERNATIVA: Faça upload manual do dataset")
    print("   Execute a célula abaixo para upload manual")

In [ ]:
# ALTERNATIVA: Upload manual (descomente se necessário)
# from google.colab import files
# print("Faça upload do arquivo train.csv:")
# uploaded = files.upload()
# df = pd.read_csv('train.csv')
# print(f"✅ Dataset carregado: {df.shape[0]} registros, {df.shape[1]} colunas")

## 3. Análise Exploratória Inicial

In [ ]:
print("="*70)
print("ANÁLISE EXPLORATÓRIA DOS DADOS")
print("="*70)

# Informações básicas
print(f"\n📊 DIMENSÕES DO DATASET:")
print(f"   Registros: {df.shape[0]:,}")
print(f"   Atributos: {df.shape[1]}")

# Primeiras linhas
print("\n📋 AMOSTRA DOS DADOS:")
display(df.head())

In [ ]:
# Tipos de dados
print("🔍 TIPOS DE DADOS:")
print(df.dtypes)

In [ ]:
# Estatísticas descritivas
print("📈 ESTATÍSTICAS DESCRITIVAS:")
display(df.describe())

In [ ]:
# Valores ausentes
print("❓ VALORES AUSENTES:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Ausentes': missing, 'Percentual (%)': missing_pct})
print(missing_df[missing_df['Ausentes'] > 0])

In [ ]:
# Variável alvo
print("🎯 DISTRIBUIÇÃO DA VARIÁVEL ALVO (satisfaction):")
print(df['satisfaction'].value_counts())
print(f"\nProporção:")
print(df['satisfaction'].value_counts(normalize=True).round(4) * 100)

# Visualização
plt.figure(figsize=(8, 5))
colors = ['#e74c3c', '#2ecc71']
df['satisfaction'].value_counts().plot(kind='bar', color=colors, edgecolor='black')
plt.title('Distribuição da Variável Alvo - Satisfaction', fontsize=14, fontweight='bold')
plt.xlabel('Satisfação', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Pré-Processamento dos Dados

In [ ]:
print("="*70)
print("PRÉ-PROCESSAMENTO DOS DADOS")
print("="*70)

# Criar cópia para processamento
df_processed = df.copy()

# Remover colunas desnecessárias
cols_to_drop = ['Unnamed: 0', 'id']
df_processed = df_processed.drop(columns=[col for col in cols_to_drop if col in df_processed.columns])
print(f"✅ Colunas removidas: {cols_to_drop}")

# Tratar valores ausentes
print(f"\n📌 Valores ausentes antes do tratamento: {df_processed.isnull().sum().sum()}")
df_processed = df_processed.dropna()
print(f"📌 Valores ausentes após o tratamento: {df_processed.isnull().sum().sum()}")
print(f"📌 Registros restantes: {len(df_processed):,}")

In [ ]:
# Identificar colunas categóricas e numéricas
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"📊 Colunas Categóricas ({len(categorical_cols)}): {categorical_cols}")
print(f"📊 Colunas Numéricas ({len(numerical_cols)}): {numerical_cols}")

In [ ]:
# Encoding das variáveis categóricas
print("🔄 Aplicando Label Encoding nas variáveis categóricas...")
le = LabelEncoder()
encoding_map = {}

for col in categorical_cols:
    df_processed[col] = le.fit_transform(df_processed[col])
    encoding_map[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"   {col}: {encoding_map[col]}")

In [ ]:
# Separar features e target
X = df_processed.drop('satisfaction', axis=1)
y = df_processed['satisfaction']

feature_names = X.columns.tolist()
print(f"\n✅ Features disponíveis ({len(feature_names)}):")
for i, feat in enumerate(feature_names, 1):
    print(f"   {i:2d}. {feat}")

print(f"\n✅ Pré-processamento concluído!")
print(f"   Shape de X: {X.shape}")
print(f"   Shape de y: {y.shape}")

## 5. Feature Selection - Mutual Information

In [ ]:
print("="*70)
print("FEATURE SELECTION - MUTUAL INFORMATION")
print("="*70)

print("""
📖 MUTUAL INFORMATION (Informação Mútua):
   • Mede a dependência entre duas variáveis
   • Captura relações NÃO-LINEARES entre features e target
   • Valor 0 indica independência total
   • Valores maiores indicam maior dependência/relevância
   • Vantagem: Detecta relações complexas que ANOVA não captura
""")

In [ ]:
# Calcular Mutual Information para todas as features
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({
    'Feature': feature_names,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False).reset_index(drop=True)

print("🏆 RANKING DE FEATURES - MUTUAL INFORMATION:")
display(mi_df)

In [ ]:
# Visualização Mutual Information
plt.figure(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(mi_df)))
bars = plt.barh(mi_df['Feature'], mi_df['MI_Score'], color=colors)
plt.xlabel('Mutual Information Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Feature Selection - Mutual Information\nAirline Passenger Satisfaction', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Adicionar valores nas barras
for bar, score in zip(bars, mi_df['MI_Score']):
    plt.text(score + 0.005, bar.get_y() + bar.get_height()/2, 
             f'{score:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# SelectKBest com Mutual Information - K=10 e K=15
print("-"*50)
print("SELECTKBEST COM MUTUAL INFORMATION")
print("-"*50)

# K = 10
selector_mi_10 = SelectKBest(score_func=mutual_info_classif, k=10)
selector_mi_10.fit(X, y)
mi_features_10 = X.columns[selector_mi_10.get_support()].tolist()
print(f"\n🔟 Top 10 Features (MI):")
for i, feat in enumerate(mi_features_10, 1):
    print(f"   {i:2d}. {feat}")

# K = 15
selector_mi_15 = SelectKBest(score_func=mutual_info_classif, k=15)
selector_mi_15.fit(X, y)
mi_features_15 = X.columns[selector_mi_15.get_support()].tolist()
print(f"\n1️⃣5️⃣ Top 15 Features (MI):")
for i, feat in enumerate(mi_features_15, 1):
    print(f"   {i:2d}. {feat}")

## 6. Feature Selection - ANOVA (F-CLASSIF)

In [ ]:
print("="*70)
print("FEATURE SELECTION - ANOVA (F-CLASSIF)")
print("="*70)

print("""
📖 ANOVA F-VALUE (f_classif):
   • Análise de Variância para classificação
   • Mede a variância ENTRE grupos vs. DENTRO dos grupos
   • Assume relações LINEARES
   • F-value alto indica feature discriminativa
   • P-value baixo indica significância estatística
   • Vantagem: Rápido e estatisticamente interpretável
""")

In [ ]:
# Calcular ANOVA F-scores
f_scores, p_values = f_classif(X, y)

anova_df = pd.DataFrame({
    'Feature': feature_names,
    'F_Score': f_scores,
    'P_Value': p_values
}).sort_values('F_Score', ascending=False).reset_index(drop=True)

print("🏆 RANKING DE FEATURES - ANOVA F-VALUE:")
display(anova_df)

In [ ]:
# Visualização ANOVA
plt.figure(figsize=(12, 8))
colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(anova_df)))
bars = plt.barh(anova_df['Feature'], anova_df['F_Score'], color=colors)
plt.xlabel('ANOVA F-Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Feature Selection - ANOVA F-Value\nAirline Passenger Satisfaction', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Adicionar valores nas barras
for bar, score in zip(bars, anova_df['F_Score']):
    plt.text(score + max(anova_df['F_Score'])*0.01, bar.get_y() + bar.get_height()/2, 
             f'{score:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# SelectKBest com ANOVA - K=10 e K=15
print("-"*50)
print("SELECTKBEST COM ANOVA (F-CLASSIF)")
print("-"*50)

# K = 10
selector_anova_10 = SelectKBest(score_func=f_classif, k=10)
selector_anova_10.fit(X, y)
anova_features_10 = X.columns[selector_anova_10.get_support()].tolist()
print(f"\n🔟 Top 10 Features (ANOVA):")
for i, feat in enumerate(anova_features_10, 1):
    print(f"   {i:2d}. {feat}")

# K = 15
selector_anova_15 = SelectKBest(score_func=f_classif, k=15)
selector_anova_15.fit(X, y)
anova_features_15 = X.columns[selector_anova_15.get_support()].tolist()
print(f"\n1️⃣5️⃣ Top 15 Features (ANOVA):")
for i, feat in enumerate(anova_features_15, 1):
    print(f"   {i:2d}. {feat}")

## 7. Comparação dos Métodos - Diagrama de Venn

In [ ]:
print("="*70)
print("COMPARAÇÃO DOS MÉTODOS - DIAGRAMA DE VENN")
print("="*70)

# Converter para sets
mi_set_10 = set(mi_features_10)
mi_set_15 = set(mi_features_15)
anova_set_10 = set(anova_features_10)
anova_set_15 = set(anova_features_15)

In [ ]:
# Criar figura com subplots para os diagramas de Venn
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Diagrama de Venn - K=10
plt.subplot(1, 2, 1)
venn_10 = venn2([mi_set_10, anova_set_10], 
                set_labels=('Mutual Information', 'ANOVA'),
                set_colors=('#3498db', '#e74c3c'),
                alpha=0.7)
plt.title('Comparação de Features Selecionadas\nK = 10', fontsize=13, fontweight='bold')

# Diagrama de Venn - K=15
plt.subplot(1, 2, 2)
venn_15 = venn2([mi_set_15, anova_set_15], 
                set_labels=('Mutual Information', 'ANOVA'),
                set_colors=('#3498db', '#e74c3c'),
                alpha=0.7)
plt.title('Comparação de Features Selecionadas\nK = 15', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Análise detalhada das diferenças
print("-"*50)
print("ANÁLISE DETALHADA DAS DIFERENÇAS")
print("-"*50)

print("\n📊 K = 10 FEATURES:")
print(f"   • Apenas MI: {mi_set_10 - anova_set_10}")
print(f"   • Apenas ANOVA: {anova_set_10 - mi_set_10}")
print(f"   • Ambos os métodos: {mi_set_10 & anova_set_10}")
print(f"   • Concordância: {len(mi_set_10 & anova_set_10)}/10 ({len(mi_set_10 & anova_set_10)/10*100:.1f}%)")

print("\n📊 K = 15 FEATURES:")
print(f"   • Apenas MI: {mi_set_15 - anova_set_15}")
print(f"   • Apenas ANOVA: {anova_set_15 - mi_set_15}")
print(f"   • Ambos os métodos: {mi_set_15 & anova_set_15}")
print(f"   • Concordância: {len(mi_set_15 & anova_set_15)}/15 ({len(mi_set_15 & anova_set_15)/15*100:.1f}%)")

In [ ]:
# Tabela comparativa de rankings
print("-"*50)
print("TABELA COMPARATIVA DE RANKINGS")
print("-"*50)

comparison_df = pd.DataFrame({
    'Rank': range(1, len(feature_names)+1),
    'MI_Feature': mi_df['Feature'].values,
    'MI_Score': mi_df['MI_Score'].round(4).values,
    'ANOVA_Feature': anova_df['Feature'].values,
    'ANOVA_F_Score': anova_df['F_Score'].round(2).values
})
display(comparison_df)

## 8. Avaliação do Modelo Preditivo

In [ ]:
print("="*70)
print("AVALIAÇÃO DO MODELO PREDITIVO")
print("="*70)

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"📊 Divisão dos dados:")
print(f"   Treino: {X_train.shape[0]:,} registros")
print(f"   Teste: {X_test.shape[0]:,} registros")

In [ ]:
# Função para avaliar modelo com diferentes conjuntos de features
def evaluate_model(X_train, X_test, y_train, y_test, features, method_name, k):
    """
    Avalia modelo RandomForest com um subconjunto de features
    Retorna métricas de performance
    """
    # Selecionar features
    X_train_subset = X_train[features]
    X_test_subset = X_test[features]
    
    # Treinar modelo
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train_subset, y_train)
    
    # Predições
    y_pred = model.predict(X_test_subset)
    
    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_subset, y_train, cv=cv, scoring='accuracy')
    
    return {
        'Método': method_name,
        'K': k,
        'Acurácia Teste': accuracy,
        'CV Média': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Features': features
    }

In [ ]:
# Avaliar diferentes configurações
print("\n⏳ Avaliando modelos...")
results = []

# Mutual Information - K=10
results.append(evaluate_model(X_train, X_test, y_train, y_test, 
                              mi_features_10, 'Mutual Information', 10))
print("   ✓ MI K=10 concluído")

# Mutual Information - K=15
results.append(evaluate_model(X_train, X_test, y_train, y_test, 
                              mi_features_15, 'Mutual Information', 15))
print("   ✓ MI K=15 concluído")

# ANOVA - K=10
results.append(evaluate_model(X_train, X_test, y_train, y_test, 
                              anova_features_10, 'ANOVA', 10))
print("   ✓ ANOVA K=10 concluído")

# ANOVA - K=15
results.append(evaluate_model(X_train, X_test, y_train, y_test, 
                              anova_features_15, 'ANOVA', 15))
print("   ✓ ANOVA K=15 concluído")

# Todas as features (baseline)
results.append(evaluate_model(X_train, X_test, y_train, y_test, 
                              feature_names, 'Todas Features', len(feature_names)))
print("   ✓ Baseline concluído")

print("\n✅ Avaliação concluída!")

In [ ]:
# Criar DataFrame com resultados
results_df = pd.DataFrame(results)
results_display = results_df[['Método', 'K', 'Acurácia Teste', 'CV Média', 'CV Std']].copy()
results_display['Acurácia Teste'] = results_display['Acurácia Teste'].apply(lambda x: f"{x:.4f}")
results_display['CV Média'] = results_display['CV Média'].apply(lambda x: f"{x:.4f}")
results_display['CV Std'] = results_display['CV Std'].apply(lambda x: f"{x:.4f}")

print("\n🏆 RESULTADOS DA AVALIAÇÃO - RANDOM FOREST CLASSIFIER")
print("-"*70)
display(results_display)

In [ ]:
# Visualização dos resultados
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['Acurácia Teste'], width, 
               label='Acurácia Teste', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, results_df['CV Média'], width, 
               label='CV Média (5-fold)', color='#e74c3c', alpha=0.8)

# Adicionar barras de erro para CV
ax.errorbar(x + width/2, results_df['CV Média'], yerr=results_df['CV Std'], 
            fmt='none', color='black', capsize=5)

ax.set_ylabel('Acurácia', fontsize=12)
ax.set_title('Comparação de Performance: Feature Selection Methods\nRandom Forest Classifier', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"{r['Método']}\n(K={r['K']})" for _, r in results_df.iterrows()], fontsize=10)
ax.legend(loc='lower right')
ax.set_ylim(0.85, 1.0)

# Adicionar valores nas barras
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 9. Análise Detalhada do Melhor Modelo

In [ ]:
print("="*70)
print("ANÁLISE DETALHADA - MELHOR CONFIGURAÇÃO")
print("="*70)

# Identificar melhor configuração
best_idx = results_df['CV Média'].idxmax()
best_result = results_df.iloc[best_idx]

print(f"\n🥇 MELHOR CONFIGURAÇÃO:")
print(f"   Método: {best_result['Método']}")
print(f"   K: {best_result['K']}")
print(f"   Acurácia Teste: {best_result['Acurácia Teste']:.4f}")
print(f"   CV Média: {best_result['CV Média']:.4f} (±{best_result['CV Std']:.4f})")

In [ ]:
# Treinar modelo com melhores features para análise detalhada
best_features = best_result['Features']
X_train_best = X_train[best_features]
X_test_best = X_test[best_features]

model_best = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_best.fit(X_train_best, y_train)
y_pred_best = model_best.predict(X_test_best)

# Classification Report
print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_best, 
                           target_names=['neutral or dissatisfied', 'satisfied']))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Neutral/Dissatisfied', 'Satisfied'],
            yticklabels=['Neutral/Dissatisfied', 'Satisfied'])
plt.xlabel('Predito', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title(f'Matriz de Confusão\n{best_result["Método"]} (K={best_result["K"]})', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance do modelo
feature_importance = pd.DataFrame({
    'Feature': best_features,
    'Importance': model_best.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feature_importance)))[::-1]
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color=colors)
plt.xlabel('Importância', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title(f'Feature Importance - Random Forest\n{best_result["Método"]} (K={best_result["K"]})', 
          fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Resumo e Conclusões

In [ ]:
print("="*70)
print("RESUMO E CONCLUSÕES")
print("="*70)

print("""
📋 RESUMO DA ANÁLISE:

1. MÉTODOS DE FEATURE SELECTION APLICADOS:
   • Mutual Information: Captura relações não-lineares
   • ANOVA (F-classif): Baseado em variância entre grupos (linear)

2. CONFIGURAÇÕES AVALIADAS:
   • K = 10 features (Mutual Information e ANOVA)
   • K = 15 features (Mutual Information e ANOVA)
   • Baseline com todas as features
""")

print("3. CONCORDÂNCIA ENTRE MÉTODOS:")
concordance_10 = len(mi_set_10 & anova_set_10) / 10 * 100
concordance_15 = len(mi_set_15 & anova_set_15) / 15 * 100
print(f"   • K=10: {concordance_10:.1f}% das features coincidem")
print(f"   • K=15: {concordance_15:.1f}% das features coincidem")

print(f"\n4. FEATURES COMUNS (ALTA RELEVÂNCIA):")
print(f"   {mi_set_10 & anova_set_10}")

print(f"\n5. PERFORMANCE DOS MODELOS:")
for _, row in results_df.iterrows():
    print(f"   • {row['Método']} (K={row['K']}): {row['CV Média']:.4f} ± {row['CV Std']:.4f}")

print(f"""
6. CONCLUSÕES:
   • A seleção de features pode manter performance comparável ao uso de todas as features
   • Redução de dimensionalidade melhora interpretabilidade e reduz custo computacional
   • {best_result['Método']} com K={best_result['K']} apresentou melhor resultado
   • Features consistentemente selecionadas por ambos os métodos são mais confiáveis
   • Mutual Information é preferível quando há suspeita de relações não-lineares
   • ANOVA é mais rápido e fornece interpretação estatística (p-value)
""")

print("="*70)
print("FIM DA ANÁLISE - FEATURE SELECTION AIRLINE PASSENGER SATISFACTION")
print("="*70)